In [3]:

from pyspark.sql import functions as F
import re

SOURCE_FILES = {
    "online_retail_2009_2010.csv":
        "Files/bronze/source/online_retail_2009_2010.csv",
    "online_retail_2010_2011.csv":
        "Files/bronze/source/online_retail_2010_2011.csv",
}

def normalize_column_name(name):
    """Convert source column names to lowercase snake_case."""
    name = name.replace("\ufeff", "").strip()
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    name = re.sub(r"[^A-Za-z0-9]+", "_", name)
    return name.strip("_").lower()

def read_retail_csv(file_name, file_path):
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("encoding", "UTF-8")
        .csv(file_path)
    )

    for old_name in df.columns:
        new_name = normalize_column_name(old_name)
        if old_name != new_name:
            df = df.withColumnRenamed(old_name, new_name)

    df = (
        df
        .withColumn("_source_file", F.lit(file_name))
        .withColumn("_ingested_at", F.current_timestamp())
    )

    return df

print("Bronze ingestion configuration loaded.")
print("Files to process:", list(SOURCE_FILES.keys()))


StatementMeta(, 2738d98e-b59d-4722-aaf7-9bb46bba6234, 5, Finished, Available, Finished, False)

Bronze ingestion configuration loaded.
Files to process: ['online_retail_2009_2010.csv', 'online_retail_2010_2011.csv']


In [10]:

# Update the input paths to match your Lakehouse folder

SOURCE_FILES = {
    "online_retail_2009_2010.csv":
        "Files/Source/online_retail_2009_2010.csv",

    "online_retail_2010_2011.csv":
        "Files/Source/online_retail_2010_2011.csv",
}

print("Bronze ingestion configuration loaded.")
print("Files to process:")

for file_name, file_path in SOURCE_FILES.items():
    print(f"{file_name} -> {file_path}")



StatementMeta(, 2738d98e-b59d-4722-aaf7-9bb46bba6234, 12, Finished, Available, Finished, False)

Bronze ingestion configuration loaded.
Files to process:
online_retail_2009_2010.csv -> Files/Source/online_retail_2009_2010.csv
online_retail_2010_2011.csv -> Files/Source/online_retail_2010_2011.csv


In [11]:

for file_name, file_path in SOURCE_FILES.items():
    print(f"Reading {file_name}...")

    df = read_retail_csv(file_name, file_path)

    if not df.columns:
        raise ValueError(f"No columns found in {file_name}")

    row_count = df.count()

    if row_count == 0:
        raise ValueError(f"No data rows found in {file_name}")

    if "2009_2010" in file_name:
        table_name = "bronze_online_retail_2009_2010"
    else:
        table_name = "bronze_online_retail_2010_2011"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    print(f"Rows loaded: {row_count:,}")
    print(f"Bronze table created: {table_name}")

print("Bronze ingestion completed.")


StatementMeta(, 2738d98e-b59d-4722-aaf7-9bb46bba6234, 13, Finished, Available, Finished, False)

Reading online_retail_2009_2010.csv...
Rows loaded: 525,461
Bronze table created: bronze_online_retail_2009_2010
Reading online_retail_2010_2011.csv...
Rows loaded: 541,910
Bronze table created: bronze_online_retail_2010_2011
Bronze ingestion completed.


In [12]:

df_2009 = spark.table("bronze_online_retail_2009_2010")
df_2010 = spark.table("bronze_online_retail_2010_2011")

df_all = df_2009.unionByName(
    df_2010,
    allowMissingColumns=True
)

(
    df_all.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_online_retail_all")
)

print("Combined Bronze table created.")
print("Columns:", df_all.columns)


StatementMeta(, 2738d98e-b59d-4722-aaf7-9bb46bba6234, 14, Finished, Available, Finished, False)

Combined Bronze table created.
Columns: ['invoice', 'stock_code', 'description', 'quantity', 'invoice_date', 'price', 'customer_id', 'country', '_source_file', '_ingested_at']


In [13]:

tables = [
    "bronze_online_retail_2009_2010",
    "bronze_online_retail_2010_2011",
    "bronze_online_retail_all",
]

for table_name in tables:
    df = spark.table(table_name)

    print(f"\nTable: {table_name}")
    print(f"Rows: {df.count():,}")
    print("Columns:", df.columns)
    display(df.limit(5))


StatementMeta(, 2738d98e-b59d-4722-aaf7-9bb46bba6234, 15, Finished, Available, Finished, False)


Table: bronze_online_retail_2009_2010
Rows: 525,461
Columns: ['invoice', 'stock_code', 'description', 'quantity', 'invoice_date', 'price', 'customer_id', 'country', '_source_file', '_ingested_at']


SynapseWidget(Synapse.DataFrame, 742d66f6-4b11-4505-a7f7-756f763a842a)


Table: bronze_online_retail_2010_2011
Rows: 541,910
Columns: ['invoice', 'stock_code', 'description', 'quantity', 'invoice_date', 'price', 'customer_id', 'country', '_source_file', '_ingested_at']


SynapseWidget(Synapse.DataFrame, e02b76d4-92db-4cf5-9ca8-42f47a1d7cf2)


Table: bronze_online_retail_all
Rows: 1,067,371
Columns: ['invoice', 'stock_code', 'description', 'quantity', 'invoice_date', 'price', 'customer_id', 'country', '_source_file', '_ingested_at']


SynapseWidget(Synapse.DataFrame, 635fc1d6-8e89-455a-8183-b493980df984)